# 04 · MIP optimisation: minimise a tree surrogate with `discopt`

Train a HyperplaneTree on a quadratic, then ask discopt to find its
piecewise-linear minimum. No Pyomo or OMLT required.


In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"

import jax
jax.config.update("jax_enable_x64", True)

import numpy as np

import discopt
from jax_ldt import LinearTreeRegressor
from jax_ldt.export import embed_in_discopt_model

print("discopt", discopt.__version__)


## Train a small surrogate

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(-1.5, 1.5, size=(150, 2))
y = (X ** 2).sum(axis=1) + 0.1 * X[:, 0] * X[:, 1]
model = LinearTreeRegressor(max_depth=3, max_bins=4, min_samples_leaf=15).fit(X, y)
print(f"surrogate MAE: {float(np.mean(np.abs(model.predict(X) - y))):.4f}, leaves: {model.num_leaves}")


## Embed the tree in a `discopt.Model`

`embed_in_discopt_model` adds:
1. lifted-feature equality constraints,
2. one binary `z_k` per leaf with `Σ z_k = 1`,
3. big-M routing constraints for every internal split,
4. bilinear leaf regressions linearised with big-M.

It returns a discopt expression you can use as an objective or constraint.


In [ ]:
m = discopt.Model("min-tree")
x = m.continuous("x", shape=(2,), lb=-1.5, ub=1.5)
y_expr = embed_in_discopt_model(model, m, x, big_m=10.0)
m.minimize(y_expr[0])
result = m.solve()
x_star = np.asarray(result.x["x"])
print("argmin:", x_star.round(3))
print("min y :", float(result.objective))
print("in-sample y range:", y.min(), "...", y.max())


The recovered minimum is at or below the smallest training point, since discopt is solving the surrogate's piecewise-linear minimum exactly (not an interpolation). For a non-trivial example with constraints or mixed-integer variables, just add them to `m` alongside `x`.
